<a href="https://colab.research.google.com/github/leoncana/CreativeHub-DataEngineeringModule1/blob/main/homework/hw05_spark_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Homework 5 — Analysing two weeks of plays

About 60 minutes. You analyse **all 14 days** of raw play events (1-14 September) with Spark.

In [1]:
# Run me first: finds the course data, or downloads it when running in Google Colab.
import subprocess
from pathlib import Path

COURSE_REPO = "https://github.com/alinafazliu/data-engineering-modul-1.git"


def find_data():
    for folder in [Path.cwd(), *Path.cwd().parents, Path("/content/course")]:
        for candidate in [folder / "data", folder / "month_course" / "data"]:
            if (candidate / "reference" / "tracks.csv").exists():
                return candidate


DATA = find_data()
if DATA is None:
    subprocess.run(["git", "clone", "--depth", "1", "--quiet", COURSE_REPO, "/content/course"], check=True)
    DATA = find_data()
print("Course data:", DATA)

Course data: /content/course/data


In [2]:
# Start Spark. Colab already has PySpark; anywhere else we install it first.
import importlib.util
import os
import subprocess
import sys

if importlib.util.find_spec("pyspark") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "pyspark>=3.5,<3.6"], check=True)
os.environ["PYSPARK_PYTHON"] = sys.executable  # Spark's workers must use the same Python as the notebook

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("wavely")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.ansi.enabled", "false")  # same behaviour on Spark 3.5 and 4.x: bad casts give null
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "is ready")

Spark 4.0.5 is ready


### Exercise 1 — Load and clean
1. Read **all** files `DATA / "raw" / "plays_*.csv"` with the schema below.
2. Remove duplicate events (by `event_id`) and rows with `ms_played < 0`.
3. Add a column `play_date` (`F.to_date("event_ts")`).

Store the result in `plays`.

In [13]:
PLAYS_SCHEMA = (
    "event_id STRING, event_ts TIMESTAMP, user_id STRING, track_id STRING, ms_played INT, "
    "device STRING, country STRING, skipped BOOLEAN, shuffle BOOLEAN"
)

# YOUR CODE HERE
plays = (
    spark.read.csv(str(DATA / "raw" / "plays_*.csv"), header=True, schema=PLAYS_SCHEMA)
    .dropDuplicates(["event_id"])  # the app sometimes sends an event twice
    .filter(F.col("ms_played") >= 0)
    .withColumn('play_date', F.to_date("event_ts"))
)

print(plays.count(), "plays on", plays.select("play_date").distinct().count(), "days")

59801 plays on 14 days


### Exercise 2 — Device share
For each device: number of plays and `share` = percent of all plays (rounded to 1 decimal).
Hint: compute `total = plays.count()` first and use it in `withColumn`.

In [15]:
# YOUR CODE HERE
plays.show()
total=plays.count()

device_shares = (
    plays
    .groupBy('device')
    .count()
    .withColumn('share', F.round((F.col('count')/total)*100, 1))
).show()

+------------+-------------------+-------+--------+---------+-------------+-------+-------+-------+----------+
|    event_id|           event_ts|user_id|track_id|ms_played|       device|country|skipped|shuffle| play_date|
+------------+-------------------+-------+--------+---------+-------------+-------+-------+-------+----------+
|000604a07778|2026-09-08 21:07:28|  U0139|   T0087|     9392|       mobile|     PL|   true|  false|2026-09-08|
|00067568a5c3|2026-09-12 10:27:59|  U0248|   T0140|   328000|smart_speaker|     SK|  false|  false|2026-09-12|
|00086ef381dd|2026-09-12 11:42:02|  U0163|   T0033|     4919|       mobile|     AT|   true|   true|2026-09-12|
|000adec8e137|2026-09-06 06:31:05|  U0661|   T0031|    33746|      desktop|     AT|  false|  false|2026-09-06|
|000d000096f6|2026-09-11 21:56:16|  U0762|   T0020|   181000|       mobile|     DE|  false|  false|2026-09-11|
|00106b011fea|2026-09-07 22:57:16|  U0563|   T0013|   201000|      desktop|     CZ|  false|   true|2026-09-07|
|

### Exercise 3 — When do people listen?
`F.hour(ts)` gives the hour, `F.dayofweek(ts)` gives 1 (Sunday) ... 7 (Saturday).
1. Add `hour` and `is_weekend` (`True` for Saturday and Sunday).
2. Count plays per `is_weekend` and `hour`.
3. Show the 3 busiest hours on weekdays and the 3 busiest on weekends. Are they different?

In [20]:
# YOUR CODE HERE

plays_time = (
    plays.withColumn('hour', F.hour('event_ts'))
    .withColumn('is_weekend', F.dayofweek('event_ts').isin(1,7))
)

hourly_plays = (
    plays_time.groupBy('is_weekend', 'hour')
    .agg(
        F.count('*').alias('plays')
    )
)

hourly_plays.filter(F.col('is_weekend')== False).orderBy(F.desc('plays')).show(3)
hourly_plays.filter(F.col('is_weekend')== True).orderBy(F.desc('plays')).show(3)


+----------+----+-----+
|is_weekend|hour|plays|
+----------+----+-----+
|     false|  18| 3332|
|     false|  19| 3216|
|     false|  20| 3007|
+----------+----+-----+
only showing top 3 rows
+----------+----+-----+
|is_weekend|hour|plays|
+----------+----+-----+
|      true|  19| 1480|
|      true|  20| 1462|
|      true|  21| 1351|
+----------+----+-----+
only showing top 3 rows


### Exercise 4 — Does shuffle make people skip?
Compute the skip rate (percent, 1 decimal) for `shuffle = true` and `shuffle = false`.

In [ ]:
# YOUR CODE HERE

total_by_shuffle = (
    plays.groupBy('shuffle')
    .count()
    .withColumnRenamed('count', 'total_plays')
)

skipped_by_shuffle = (
    plays.filter(F.col('skipped')==True)
    .groupBy()
)

### Exercise 5 — Explicit content by plan
Load `tracks.csv` and `users.csv`. For each `plan`, what percentage of plays are of **explicit** tracks?

In [23]:
# YOUR CODE HERE
tracks = spark.read.csv(str(DATA / "reference" / "tracks.csv"), header=True, inferSchema=True)
users = spark.read.csv(str(DATA / "reference" / "users.csv"), header=True, inferSchema=True)
explicit_by_plan = (
    plays.join(tracks.select('track_id', 'explicit'), on = 'track_id', how='inner')
    .join(users.select('user_id', 'plan'), on= 'user_id', how='inner')
    .groupBy('plan')
    .agg(F.count('*').alias('plays'),
         F.round(F.avg(F.col('explicit').cast('int')) * 100, 1).alias('explicit_percent'))
    .orderBy('plan')
).show()

+-------+-----+----------------+
|   plan|plays|explicit_percent|
+-------+-----+----------------+
|   free|25678|            15.2|
|premium|34123|            15.3|
+-------+-----+----------------+



### Exercise 6 — Save a result table
1. Build `daily_country`: per `play_date` and `country`: `plays` and `hours` (rounded to 2 decimals).
2. Write it as Parquet to `DATA.parent / "output" / "daily_country"`, **partitioned by country**, with mode `overwrite`.
3. Read back only Austria (`country=AT` sub-folder or filter after reading) and show it sorted by date.

In [ ]:
# YOUR CODE HERE

daily_country=(
    plays.groupBy('play_date', 'country')
    .agg(
        F.count('*').alias('plays'),
        F.round(F.sum('ms_played')/3600000, 2).alias('hours')
    )
)



### Exercise 7 (Stretch) — Top 3 tracks per genre
Show, for every genre, the 3 most played tracks with their title and artist name. Use a window function with `F.row_number()`.

In [26]:
# YOUR CODE HERE

from pyspark.sql.window import Window

artists = spark.read.csv(str(DATA / "reference" / "artists.csv"), header=True, inferSchema=True)

track_counts = (
    plays.groupBy('track_id')
    .agg(
        F.count('*').alias('plays')
    )

)

tracks_with_count = (
    track_counts.join(tracks.select('track_id', 'title', 'artist_id', 'genre'), on = 'track_id')
    .join(artists.select('artist_id', 'artist_name'), on='artist_id')
)

genre_window = Window.partitionBy('genre').orderBy(F.desc('plays'), 'track_id')

top_3_tracks = (
    tracks_with_count
    .withColumn('rank', F.row_number().over(genre_window))
    .filter(F.col('rank')<=3)
    .select('genre', 'rank', 'track_id', 'title', 'artist_name', 'plays')
    .orderBy('genre', 'rank')
)
top_3_tracks.show()
top_3_tracks.count()


+----------+----+--------+---------------+----------------+-----+
|     genre|rank|track_id|          title|     artist_name|plays|
+----------+----+--------+---------------+----------------+-----+
| classical|   1|   T0183| Northern River|Ludovico Einaudi|  454|
| classical|   2|   T0185|    Velvet Fire|Ludovico Einaudi|  442|
| classical|   3|   T0186|  Velvet Garden|Ludovico Einaudi|  437|
|electronic|   1|   T0121|    Neon Hearts|    David Guetta|  710|
|electronic|   2|   T0118|   Faded Hearts|    David Guetta|  637|
|electronic|   3|   T0130|   Paper Garden|   Calvin Harris|  558|
|   hip-hop|   1|   T0095| Summer Letters|           Drake|  875|
|   hip-hop|   2|   T0078|Electric Garden|  Kendrick Lamar|  829|
|   hip-hop|   3|   T0092|   Hidden Waves|           Drake|  798|
|     latin|   1|   T0061|   Faded Lights|       Bad Bunny| 1115|
|     latin|   2|   T0060|Endless Thunder|       Bad Bunny|  735|
|     latin|   3|   T0066| Summer Letters|       Bad Bunny|  732|
|       po

18